# External Predictor Faithful Feature Rebuild

This notebook rebuilds Wikipedia and competition variables before making a final call on external residual predictors.

It keeps the primary forecast problem as frozen-baseline opening-weekend residuals:

`u_OW = log(actual OW / baseline OW)`

It also adds secondary daily-shape diagnostics:

- after Friday: `log(Sat/Fri)` residual;
- after Friday: `log(Sun/Fri)` residual;
- after Saturday: `log(Sun/Sat)` residual.

The feature rebuild follows two guardrails:

- rolling-origin transformations only, including `WikiEngagementRolling` and `WikiViewsSurprise`;
- competition variables include same-week new openers plus a labeled holdover attraction proxy built from prior known gross, release age, broad audience segment decay, and current theater count when available.

It also tests Path B: a pre-weekend pseudo-prior model using Wiki activity, theaters, and release corridor to fill or augment missing consensus estimates.


In [ ]:
import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "pm-box-office-matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", os.path.join(tempfile.gettempdir(), "pm-box-office-cache"))

repo_root = Path.cwd().parent if Path.cwd().name == "eda" else Path.cwd()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

plt.style.use("default")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

## Parameters

In [ ]:
DIAGNOSTICS_DIR = repo_root / "data" / "diagnostics"
PREDICTIONS_DIR = repo_root / "data" / "predictions"

AF2_COMPONENT_PATH = DIAGNOSTICS_DIR / "af2_internal_calibration_predictions.csv"
AS0_COMPONENT_PATH = DIAGNOSTICS_DIR / "as0_corridor_quantile_interval_predictions.csv"
STABILIZED_INTERVAL_PATH = DIAGNOSTICS_DIR / "stabilized_interval_model_predictions.csv"
WIKI_PANEL_PATH = DIAGNOSTICS_DIR / "late_forecast_movie_origin_panel.csv"
SHAPE_BASE_PATH = DIAGNOSTICS_DIR / "weekend_shape_base.csv"
CONSENSUS_BENCHMARK_PATH = DIAGNOSTICS_DIR / "late_forecast_consensus_benchmark.csv"

MIN_TRAIN_OBS = 25
MIN_SCREEN_OBS = 15
RIDGE_LAMBDA = 1e-6

WIKI_MODELS = {
    "W1_wiki_views_surprise": ["WikiViewsSurprise"],
    "W2_wiki_engagement_rolling": ["WikiEngagementRolling"],
    "W3_wiki_surprise_plus_growth": ["WikiViewsSurprise", "WikiGrowth"],
}

COMPETITION_MODELS = {
    "C1_relative_competition": ["RelativeCompetition"],
    "C2_direct_competition": ["DirectCompetition"],
    "C3_market_strength_plus_relative": ["MarketStrength", "RelativeCompetition"],
    "C4_relative_new_competition": ["RelativeNewCompetition"],
    "C5_direct_new_competition": ["DirectNewCompetition"],
    "C6_max_new_threat": ["MaxThreatNewOnly"],
}

MODEL_SPECS = {"M0_baseline": [], **WIKI_MODELS, **COMPETITION_MODELS}

SCREEN_FEATURES = [
    "WikiViewsSurprise", "WikiEngagementRolling", "WikiGrowth",
    "MarketStrengthNewOnly", "CompetitorPressureNewOnly", "RelativeNewCompetition",
    "MaxThreatNewOnly", "DirectNewCompetition",
    "MarketStrength", "CompetitorPressure", "RelativeCompetition", "MaxThreat", "DirectCompetition",
    "NewAttractionSum", "HoldoverAttractionSum", "WeightedNewAttractionSum", "WeightedHoldoverAttractionSum",
]

np.random.seed(17)


## Load Frozen Baselines And Targets

After-Friday uses `AF2+C0` from the internal calibration artifact. After-Saturday uses `AS0` from the corridor quantile artifact. Opening-weekend intervals are read from the stabilized interval file only for baseline validation checks.

In [ ]:
def read_required_csv(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {path}")
    return pd.read_csv(path)


def safe_log_ratio(num, den):
    num = pd.to_numeric(num, errors="coerce")
    den = pd.to_numeric(den, errors="coerce")
    out = pd.Series(np.nan, index=num.index, dtype="float64")
    mask = num.gt(0) & den.gt(0)
    out.loc[mask] = np.log(num.loc[mask] / den.loc[mask])
    return out


def load_component_baselines():
    af = read_required_csv(AF2_COMPONENT_PATH)
    af = af.loc[af["calibration_model"].eq("AF2+C0")].copy()
    af["origin"] = "after_friday"
    af["baseline_model"] = "AF2+C0"

    as0 = read_required_csv(AS0_COMPONENT_PATH)
    as0 = as0.loc[as0["calibration_model"].eq("AS0")].copy()
    as0["origin"] = "after_saturday"
    as0["baseline_model"] = "AS0"

    common_cols = sorted(set(af.columns).intersection(set(as0.columns)))
    base = pd.concat([af[common_cols], as0[common_cols]], ignore_index=True, sort=False)
    base["opening_weekend_start"] = pd.to_datetime(base["opening_weekend_start"], errors="coerce")
    numeric_cols = [
        "release_run_id", "friday_gross_usd", "saturday_gross_usd", "sunday_gross_usd",
        "opening_weekend_gross_usd", "pred_saturday_gross_usd", "pred_sunday_gross_usd",
        "pred_opening_weekend_gross_usd", "calibrated_pred_ow", "opening_weekend_theaters",
    ]
    for col in numeric_cols:
        if col in base.columns:
            base[col] = pd.to_numeric(base[col], errors="coerce")
    base["baseline_pred_ow"] = pd.to_numeric(base.get("calibrated_pred_ow"), errors="coerce")
    base["baseline_pred_ow"] = base["baseline_pred_ow"].where(base["baseline_pred_ow"].notna(), pd.to_numeric(base.get("pred_opening_weekend_gross_usd"), errors="coerce"))
    base["u_OW"] = safe_log_ratio(base["opening_weekend_gross_usd"], base["baseline_pred_ow"])
    base["baseline_y_sat_fri"] = safe_log_ratio(base["pred_saturday_gross_usd"], base["friday_gross_usd"])
    base["actual_y_sat_fri"] = safe_log_ratio(base["saturday_gross_usd"], base["friday_gross_usd"])
    base["u_SatFri"] = base["actual_y_sat_fri"] - base["baseline_y_sat_fri"]
    base["baseline_y_sun_fri"] = safe_log_ratio(base["pred_sunday_gross_usd"], base["friday_gross_usd"])
    base["actual_y_sun_fri"] = safe_log_ratio(base["sunday_gross_usd"], base["friday_gross_usd"])
    base["u_SunFri"] = base["actual_y_sun_fri"] - base["baseline_y_sun_fri"]
    base["baseline_y_sun_sat"] = safe_log_ratio(base["pred_sunday_gross_usd"], base["saturday_gross_usd"])
    base["actual_y_sun_sat"] = safe_log_ratio(base["sunday_gross_usd"], base["saturday_gross_usd"])
    base["u_SunSat"] = base["actual_y_sun_sat"] - base["baseline_y_sun_sat"]
    return base

baseline_components = load_component_baselines()

target_rows = []
for _, row in baseline_components.iterrows():
    target_rows.append({**row.to_dict(), "target_name": "OW", "target_residual": row["u_OW"], "target_family": "ow"})
    if row["origin"] == "after_friday":
        target_rows.append({**row.to_dict(), "target_name": "SatFri", "target_residual": row["u_SatFri"], "target_family": "daily_shape"})
        target_rows.append({**row.to_dict(), "target_name": "SunFri", "target_residual": row["u_SunFri"], "target_family": "daily_shape"})
    if row["origin"] == "after_saturday":
        target_rows.append({**row.to_dict(), "target_name": "SunSat", "target_residual": row["u_SunSat"], "target_family": "daily_shape"})

target_panel = pd.DataFrame(target_rows)

target_summary = target_panel.groupby(["origin", "baseline_model", "target_name", "target_family"]).agg(
    n=("target_residual", lambda s: s.notna().sum()),
    mean_residual=("target_residual", "mean"),
    mae_log=("target_residual", lambda s: s.abs().mean()),
).reset_index()
display(target_summary)

if STABILIZED_INTERVAL_PATH.exists():
    intervals = read_required_csv(STABILIZED_INTERVAL_PATH)
    interval_check = intervals.loc[
        ((intervals["model"].eq("AF2+C0")) & intervals["interval_method"].eq("I0"))
        | ((intervals["model"].eq("AS0")) & intervals["interval_method"].eq("I2"))
    ].groupby(["model", "interval_method"]).agg(
        n=("release_run_id", "size"),
        OW80=("cover80", "mean"),
        OW95=("cover95", "mean"),
        MAE_log=("log_error", lambda s: s.abs().mean()),
    ).reset_index()
    display(interval_check)

## Build Raw Wiki Features

The raw feature table stores log-transformed activity variables only. Rolling engagement and Wiki surprise are computed later with prior-release information only.

In [ ]:
def build_wiki_raw_features(panel_path, shape_path, focal_ids=None):
    focal_ids = None if focal_ids is None else set(pd.to_numeric(pd.Series(list(focal_ids)), errors="coerce").dropna())
    if panel_path.exists():
        panel = pd.read_csv(panel_path)
        panel["release_run_id"] = pd.to_numeric(panel["release_run_id"], errors="coerce")
        panel["forecast_origin_day"] = pd.to_numeric(panel.get("forecast_origin_day"), errors="coerce")
        pre = panel.loc[panel["forecast_origin_day"].le(-1)].copy()
        if pre.empty:
            pre = panel.copy()
        pre["origin_rank"] = (pre["forecast_origin_day"] + 1).abs()
        pre = pre.sort_values(["release_run_id", "origin_rank", "forecast_origin_day"])
        pre = pre.drop_duplicates("release_run_id", keep="first")
        keep = ["release_run_id", "v_cume", "u_cume", "e_cume", "r_cume", "v_recent7", "forecast_origin_day", "forecast_origin_date"]
        wiki = pre[[c for c in keep if c in pre.columns]].copy()
    else:
        wiki = pd.DataFrame(columns=["release_run_id"])

    if shape_path.exists():
        shape = pd.read_csv(shape_path, usecols=lambda c: c in ["release_run_id", "wiki_views_cume_to_m1"])
        shape["release_run_id"] = pd.to_numeric(shape["release_run_id"], errors="coerce")
        shape = shape.drop_duplicates("release_run_id")
        if focal_ids is not None:
            shape = shape.loc[shape["release_run_id"].isin(focal_ids)].copy()
        wiki = shape.merge(wiki, on="release_run_id", how="left") if not wiki.empty else shape
        if "v_cume" not in wiki.columns:
            wiki["v_cume"] = np.nan
        wiki["v_cume"] = wiki["v_cume"].where(wiki["v_cume"].notna(), pd.to_numeric(wiki.get("wiki_views_cume_to_m1"), errors="coerce"))
    elif focal_ids is not None:
        wiki = wiki.loc[wiki["release_run_id"].isin(focal_ids)].copy()

    for col in ["v_cume", "u_cume", "e_cume", "r_cume", "v_recent7"]:
        if col not in wiki.columns:
            wiki[col] = np.nan
        wiki[col] = pd.to_numeric(wiki[col], errors="coerce").clip(lower=0)

    wiki["WikiViews"] = np.log1p(wiki["v_cume"])
    wiki["WikiUsers"] = np.log1p(wiki["u_cume"])
    wiki["WikiEdits"] = np.log1p(wiki["e_cume"])
    wiki["WikiRigor"] = np.log1p(wiki["r_cume"])
    wiki["WikiViews7"] = np.log1p(wiki["v_recent7"])
    earlier_views = (wiki["v_cume"] - wiki["v_recent7"]).clip(lower=0)
    wiki["WikiGrowth"] = wiki["WikiViews7"] - np.log1p(earlier_views)
    wiki["wiki_growth_note"] = "log1p(v_recent7)-log1p(max(v_cume-v_recent7,0)); available-data proxy for V[-7:-1] vs earlier prerelease activity"
    return wiki.drop_duplicates("release_run_id")

frozen_release_ids = baseline_components["release_run_id"].unique()
wiki_raw = build_wiki_raw_features(WIKI_PANEL_PATH, SHAPE_BASE_PATH, focal_ids=frozen_release_ids)
wiki_preview_cols = [c for c in ["release_run_id", "WikiViews", "WikiUsers", "WikiEdits", "WikiRigor", "WikiViews7", "WikiGrowth", "forecast_origin_day"] if c in wiki_raw.columns]
wiki_health = pd.DataFrame({
    "feature": ["WikiViews", "WikiUsers", "WikiEdits", "WikiRigor", "WikiViews7", "WikiGrowth"],
    "n_available": [int(wiki_raw[col].notna().sum()) for col in ["WikiViews", "WikiUsers", "WikiEdits", "WikiRigor", "WikiViews7", "WikiGrowth"]],
    "n_total_frozen_ids": len(wiki_raw),
    "missing_rate": [float(wiki_raw[col].isna().mean()) for col in ["WikiViews", "WikiUsers", "WikiEdits", "WikiRigor", "WikiViews7", "WikiGrowth"]],
})
display(wiki_health)
display(wiki_raw.loc[wiki_raw["WikiViews"].notna(), wiki_preview_cols].head())


## Build Holdover Attraction Competition Features

Holdover attraction is built as:

`A_holdover,j,t = Gross_j,lastWeekend * exp(-beta_segment * age_j)`

The local diagnostic artifacts do not contain true movie-by-week grosses or current holdover theater counts, so this implementation uses the last available gross in the artifact, currently opening-weekend gross, as `Gross_j,lastWeekend` and applies the requested segment-specific age decay. If a current theater count column is added later, the builder will scale the attraction by current/opening theater count automatically.

Then it constructs:

`RelativeCompetition = log((1 + A_focal) / (1 + sum(A_new) + sum(A_holdover)))`

and:

`DirectCompetition = log((1 + A_focal) / (1 + sum(w_ij A_new,j) + sum(w_ih A_holdover,h,t)))`

Same-week estimates are filled by a rolling theater-count prior trained only on prior releases.


In [ ]:
SEGMENT_DECAY_BETA = {
    "fan-driven": 0.90,
    "family/holiday": 0.55,
    "holiday family": 0.45,
    "doc/concert": 1.10,
    "other wide": 0.70,
    "default": 0.70,
}


def clean_segment(value):
    text = str(value).strip().lower()
    return "" if text in {"", "nan", "none"} else text


def audience_weight(focal, competitor):
    focal_segment = clean_segment(focal.get("segment_current_broad", focal.get("broad_segment", "")))
    comp_segment = clean_segment(competitor.get("segment_current_broad", competitor.get("broad_segment", "")))
    focal_genre = clean_segment(focal.get("genre", ""))
    comp_genre = clean_segment(competitor.get("genre", ""))
    if focal_segment and focal_segment == comp_segment:
        return 1.0
    if focal_genre and focal_genre == comp_genre:
        return 0.5
    return 0.25


def decay_beta_for_row(row):
    segment = clean_segment(row.get("segment_current_broad", row.get("broad_segment", "")))
    return SEGMENT_DECAY_BETA.get(segment, SEGMENT_DECAY_BETA["default"])


def fit_theater_prior(train):
    clean = train[["opening_weekend_gross_usd", "opening_weekend_theaters"]].copy()
    clean["opening_weekend_gross_usd"] = pd.to_numeric(clean["opening_weekend_gross_usd"], errors="coerce")
    clean["opening_weekend_theaters"] = pd.to_numeric(clean["opening_weekend_theaters"], errors="coerce")
    clean = clean.loc[clean["opening_weekend_gross_usd"].gt(0) & clean["opening_weekend_theaters"].gt(0)].dropna()
    if len(clean) < MIN_TRAIN_OBS:
        return None
    x = np.log1p(clean["opening_weekend_theaters"].to_numpy(dtype=float))
    y = np.log(clean["opening_weekend_gross_usd"].to_numpy(dtype=float))
    design = np.column_stack([np.ones(len(x)), x])
    coef, *_ = np.linalg.lstsq(design, y, rcond=None)
    return coef


def attraction_for_row(row, coef):
    estimate = row.get("latest_estimate_mid_usd", np.nan)
    if pd.notna(estimate) and estimate > 0:
        return float(estimate), "estimate"
    theaters = row.get("opening_weekend_theaters", np.nan)
    if coef is not None and pd.notna(theaters) and theaters > 0:
        return float(np.exp(coef[0] + coef[1] * np.log1p(theaters))), "rolling_theater_prior"
    return np.nan, "missing"


def holdover_attraction_for_row(focal_weekend, holdover):
    age_weeks = max((focal_weekend - holdover["opening_weekend_start"]).days / 7.0, 1.0)
    prior_gross = holdover.get("prior_weekend_gross_usd", np.nan)
    source = "prior_weekend_gross_usd"
    if pd.isna(prior_gross) or prior_gross <= 0:
        prior_gross = holdover.get("opening_weekend_gross_usd", np.nan)
        source = "opening_weekend_gross_decay_proxy"
    if pd.isna(prior_gross) or prior_gross <= 0:
        return np.nan, age_weeks, decay_beta_for_row(holdover), "missing"

    beta = decay_beta_for_row(holdover)
    attraction = float(prior_gross * np.exp(-beta * age_weeks))

    current_theaters = np.nan
    for col in ["current_theater_count", "current_theaters", "currenttheatercount"]:
        value = holdover.get(col, np.nan)
        if pd.notna(value) and value > 0:
            current_theaters = value
            break
    opening_theaters = holdover.get("opening_weekend_theaters", np.nan)
    if pd.notna(current_theaters) and pd.notna(opening_theaters) and opening_theaters > 0:
        attraction *= float(np.clip(current_theaters / opening_theaters, 0.0, 1.5))
        source += "+current_theater_scale"
    return attraction, age_weeks, beta, source


def build_competition_features(shape_path, focal_ids):
    shape_cols = [
        "release_run_id", "title", "opening_weekend_start", "release_year", "opening_weekend_gross_usd",
        "latest_estimate_mid_usd", "opening_weekend_theaters", "genre", "broad_segment", "segment_current_broad",
        "prior_weekend_gross_usd", "current_theater_count", "current_theaters", "currenttheatercount",
    ]
    shape = pd.read_csv(shape_path, usecols=lambda c: c in shape_cols)
    shape["release_run_id"] = pd.to_numeric(shape["release_run_id"], errors="coerce")
    shape["opening_weekend_start"] = pd.to_datetime(shape["opening_weekend_start"], errors="coerce")
    numeric_cols = [
        "opening_weekend_gross_usd", "latest_estimate_mid_usd", "opening_weekend_theaters",
        "prior_weekend_gross_usd", "current_theater_count", "current_theaters", "currenttheatercount",
    ]
    for col in numeric_cols:
        if col not in shape.columns:
            shape[col] = np.nan
        shape[col] = pd.to_numeric(shape[col], errors="coerce")
    shape = shape.dropna(subset=["release_run_id", "opening_weekend_start"]).drop_duplicates("release_run_id")
    focal_ids = set(pd.to_numeric(pd.Series(list(focal_ids)), errors="coerce").dropna())

    rows = []
    focal_rows = shape.loc[shape["release_run_id"].isin(focal_ids)].sort_values(["opening_weekend_start", "release_run_id"])
    for _, focal in focal_rows.iterrows():
        train = shape.loc[shape["opening_weekend_start"].lt(focal["opening_weekend_start"])]
        coef = fit_theater_prior(train)
        focal_attraction, focal_source = attraction_for_row(focal, coef)

        weekend = shape.loc[
            shape["opening_weekend_start"].eq(focal["opening_weekend_start"])
            & shape["release_run_id"].ne(focal["release_run_id"])
        ].copy()
        holdovers = shape.loc[shape["opening_weekend_start"].lt(focal["opening_weekend_start"])].copy()

        new_values = []
        weighted_new_values = []
        new_sources = []
        for _, comp in weekend.iterrows():
            attraction, source = attraction_for_row(comp, coef)
            if pd.notna(attraction) and attraction > 0:
                new_values.append(attraction)
                weighted_new_values.append(audience_weight(focal, comp) * attraction)
                new_sources.append(source)

        holdover_values = []
        weighted_holdover_values = []
        holdover_sources = []
        holdover_ages = []
        holdover_betas = []
        for _, holdover in holdovers.iterrows():
            attraction, age_weeks, beta, source = holdover_attraction_for_row(focal["opening_weekend_start"], holdover)
            if pd.notna(attraction) and attraction > 0:
                holdover_values.append(attraction)
                weighted_holdover_values.append(audience_weight(focal, holdover) * attraction)
                holdover_sources.append(source)
                holdover_ages.append(age_weeks)
                holdover_betas.append(beta)

        new_sum = float(np.sum(new_values)) if new_values else 0.0
        weighted_new_sum = float(np.sum(weighted_new_values)) if weighted_new_values else 0.0
        holdover_sum = float(np.sum(holdover_values)) if holdover_values else 0.0
        weighted_holdover_sum = float(np.sum(weighted_holdover_values)) if weighted_holdover_values else 0.0
        full_sum = new_sum + holdover_sum
        weighted_full_sum = weighted_new_sum + weighted_holdover_sum
        max_new = float(np.max(new_values)) if new_values else 0.0
        max_full = float(np.max(new_values + holdover_values)) if new_values or holdover_values else 0.0

        if pd.notna(focal_attraction) and focal_attraction > 0:
            market_strength_new = np.log1p(focal_attraction + new_sum)
            competitor_pressure_new = np.log1p(new_sum)
            relative_new = np.log((1 + focal_attraction) / (1 + new_sum))
            max_threat_new = np.log((1 + focal_attraction) / (1 + max_new))
            direct_new = np.log((1 + focal_attraction) / (1 + weighted_new_sum))
            market_strength = np.log1p(focal_attraction + full_sum)
            competitor_pressure = np.log1p(full_sum)
            relative = np.log((1 + focal_attraction) / (1 + full_sum))
            max_threat = np.log((1 + focal_attraction) / (1 + max_full))
            direct = np.log((1 + focal_attraction) / (1 + weighted_full_sum))
        else:
            market_strength_new = competitor_pressure_new = relative_new = max_threat_new = direct_new = np.nan
            market_strength = competitor_pressure = relative = max_threat = direct = np.nan

        rows.append({
            "release_run_id": focal["release_run_id"],
            "focal_attraction_usd": focal_attraction,
            "focal_attraction_source": focal_source,
            "same_week_competitor_count": len(weekend),
            "same_week_competitor_attraction_count": len(new_values),
            "same_week_competitor_attraction_usd": new_sum,
            "holdover_competitor_count": len(holdovers),
            "holdover_attraction_count": len(holdover_values),
            "holdover_attraction_usd": holdover_sum,
            "holdover_age_mean_weeks": float(np.mean(holdover_ages)) if holdover_ages else np.nan,
            "holdover_beta_mean": float(np.mean(holdover_betas)) if holdover_betas else np.nan,
            "NewAttractionSum": new_sum,
            "WeightedNewAttractionSum": weighted_new_sum,
            "HoldoverAttractionSum": holdover_sum,
            "WeightedHoldoverAttractionSum": weighted_holdover_sum,
            "MarketStrengthNewOnly": market_strength_new,
            "CompetitorPressureNewOnly": competitor_pressure_new,
            "RelativeNewCompetition": relative_new,
            "MaxThreatNewOnly": max_threat_new,
            "DirectNewCompetition": direct_new,
            "MarketStrength": market_strength,
            "CompetitorPressure": competitor_pressure,
            "RelativeCompetition": relative,
            "MaxThreat": max_threat,
            "DirectCompetition": direct,
            "holdover_feature_status": "opening_weekend_gross_decay_proxy" if holdover_values else "no_prior_holdovers_available",
            "competition_basis": "same_week_new_openers_plus_holdover_decay_proxy",
            "competitor_attraction_sources": ",".join(sorted(set(new_sources))) if new_sources else "none",
            "holdover_attraction_sources": ",".join(sorted(set(holdover_sources))) if holdover_sources else "none",
        })
    return pd.DataFrame(rows)

competition_features = build_competition_features(SHAPE_BASE_PATH, baseline_components["release_run_id"].unique())
competition_feature_cols = [
    "MarketStrengthNewOnly", "CompetitorPressureNewOnly", "RelativeNewCompetition",
    "MaxThreatNewOnly", "DirectNewCompetition", "MarketStrength", "CompetitorPressure",
    "RelativeCompetition", "MaxThreat", "DirectCompetition", "HoldoverAttractionSum",
]
competition_health = pd.DataFrame({
    "feature": competition_feature_cols,
    "n_available": [int(competition_features[col].notna().sum()) for col in competition_feature_cols],
    "n_nonzero_or_varying": [int(competition_features[col].nunique(dropna=True)) for col in competition_feature_cols],
    "n_total_frozen_ids": len(competition_features),
    "missing_rate": [float(competition_features[col].isna().mean()) for col in competition_feature_cols],
})
competition_source_summary = competition_features.groupby([
    "focal_attraction_source", "competitor_attraction_sources", "holdover_attraction_sources", "holdover_feature_status",
], dropna=False).size().reset_index(name="n")
segment_decay_beta = pd.DataFrame({"segment": list(SEGMENT_DECAY_BETA.keys()), "beta": list(SEGMENT_DECAY_BETA.values())})
display(segment_decay_beta)
display(competition_health)
display(competition_source_summary)
display(competition_features.loc[competition_features["holdover_attraction_count"].gt(0)].head())


## Rolling Wiki Features

`WikiEngagementRolling` and `WikiViewsSurprise` are computed chronologically by origin. For each movie, only earlier releases in the same origin panel are used to estimate scaling parameters.

In [ ]:
def fit_wiki_expectation(train):
    cols = ["WikiViews", "log_scale_prior", "log_theaters"]
    clean = train[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(clean) < MIN_TRAIN_OBS:
        return None
    x = clean[["log_scale_prior", "log_theaters"]].to_numpy(dtype=float)
    y = clean["WikiViews"].to_numpy(dtype=float)
    design = np.column_stack([np.ones(len(x)), x])
    coef, *_ = np.linalg.lstsq(design, y, rcond=None)
    return coef


def add_rolling_wiki_features(panel):
    out = panel.sort_values(["origin", "opening_weekend_start", "release_run_id"]).copy()
    out["WikiEngagementRolling"] = np.nan
    out["WikiViewsExpected"] = np.nan
    out["WikiViewsSurprise"] = np.nan
    for origin, idx in out.groupby("origin").groups.items():
        origin_rows = out.loc[idx].sort_values(["opening_weekend_start", "release_run_id"])
        for row_idx, row in origin_rows.iterrows():
            train = origin_rows.loc[origin_rows["opening_weekend_start"].lt(row["opening_weekend_start"])]
            engagement_parts = []
            for col in ["WikiUsers", "WikiEdits", "WikiRigor"]:
                train_col = pd.to_numeric(train[col], errors="coerce").dropna()
                val = row.get(col, np.nan)
                sd = train_col.std(ddof=0)
                if len(train_col) >= MIN_TRAIN_OBS and pd.notna(val) and np.isfinite(sd) and sd > 1e-12:
                    engagement_parts.append((val - train_col.mean()) / sd)
            if engagement_parts:
                out.loc[row_idx, "WikiEngagementRolling"] = float(np.mean(engagement_parts))
            coef = fit_wiki_expectation(train)
            if coef is not None and pd.notna(row.get("WikiViews")) and pd.notna(row.get("log_scale_prior")) and pd.notna(row.get("log_theaters")):
                expected = coef[0] + coef[1] * row["log_scale_prior"] + coef[2] * row["log_theaters"]
                out.loc[row_idx, "WikiViewsExpected"] = expected
                out.loc[row_idx, "WikiViewsSurprise"] = row["WikiViews"] - expected
    return out

wide_panel = baseline_components.merge(wiki_raw, on="release_run_id", how="left")
wide_panel = wide_panel.merge(competition_features, on="release_run_id", how="left")

if SHAPE_BASE_PATH.exists():
    shape_meta = pd.read_csv(
        SHAPE_BASE_PATH,
        usecols=lambda c: c in ["release_run_id", "latest_estimate_mid_usd", "opening_weekend_theaters", "genre", "broad_segment", "segment_current_broad"],
    ).drop_duplicates("release_run_id")
    wide_panel = wide_panel.merge(shape_meta, on="release_run_id", how="left", suffixes=("", "_shape"))

wide_panel["scale_prior_usd"] = pd.to_numeric(wide_panel.get("consensus_estimate_usd"), errors="coerce") if "consensus_estimate_usd" in wide_panel.columns else np.nan
wide_panel["scale_prior_usd"] = wide_panel["scale_prior_usd"].where(wide_panel["scale_prior_usd"].notna(), pd.to_numeric(wide_panel.get("latest_estimate_mid_usd"), errors="coerce"))
wide_panel["scale_prior_usd"] = wide_panel["scale_prior_usd"].where(wide_panel["scale_prior_usd"].notna(), pd.to_numeric(wide_panel.get("baseline_pred_ow"), errors="coerce"))
wide_panel["log_scale_prior"] = np.log1p(pd.to_numeric(wide_panel["scale_prior_usd"], errors="coerce").clip(lower=0))
wide_panel["log_theaters"] = np.log1p(pd.to_numeric(wide_panel.get("opening_weekend_theaters"), errors="coerce").clip(lower=0))
wide_panel = add_rolling_wiki_features(wide_panel)

feature_cols = ["release_run_id", "origin", *SCREEN_FEATURES, "competition_basis", "holdover_feature_status", "focal_attraction_source"]
feature_panel = wide_panel[[c for c in feature_cols if c in wide_panel.columns]].drop_duplicates(["origin", "release_run_id"])
display(feature_panel.head())

## Assemble Target Panel And Coverage

In [ ]:
feature_value_cols = list(dict.fromkeys([
    "origin", "release_run_id", "scale_prior_usd", "log_scale_prior", "log_theaters",
    *[c for c in SCREEN_FEATURES if c in wide_panel.columns],
    "MarketStrength", "CompetitorPressure", "RelativeCompetition", "MaxThreat", "DirectCompetition",
    "competition_basis", "holdover_feature_status", "focal_attraction_source",
]))
feature_values = wide_panel[[c for c in feature_value_cols if c in wide_panel.columns]].drop_duplicates(["origin", "release_run_id"])

modeling_panel = target_panel.drop(columns=[c for c in SCREEN_FEATURES if c in target_panel.columns], errors="ignore")
modeling_panel = modeling_panel.merge(feature_values, on=["origin", "release_run_id"], how="left")
for col in SCREEN_FEATURES:
    if col not in modeling_panel.columns:
        modeling_panel[col] = np.nan
    modeling_panel[col] = pd.to_numeric(modeling_panel[col], errors="coerce")

coverage_rows = []
for (origin, target), g in modeling_panel.groupby(["origin", "target_name"]):
    for feature in SCREEN_FEATURES:
        if feature.endswith("NewOnly") or "NewCompetition" in feature or feature in {"NewAttractionSum", "WeightedNewAttractionSum"}:
            basis = "same_week_new_openers_only"
        elif feature in {"MarketStrength", "CompetitorPressure", "RelativeCompetition", "MaxThreat", "DirectCompetition", "HoldoverAttractionSum", "WeightedHoldoverAttractionSum"}:
            basis = "new_openers_plus_holdover_decay_proxy"
        else:
            basis = "wiki"
        coverage_rows.append({
            "origin": origin,
            "target_name": target,
            "feature": feature,
            "n_total": len(g),
            "n_available": int(g[feature].notna().sum()) if feature in g.columns else 0,
            "missing_rate": float(g[feature].isna().mean()) if feature in g.columns else 1.0,
            "feature_basis": basis,
        })
faithful_feature_coverage = pd.DataFrame(coverage_rows)
display(faithful_feature_coverage.head(40))


## Compact Screens And Minimal Plots

In [ ]:
def decile_summary(df, feature, target_col="target_residual", q=10):
    clean = df[[feature, target_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(clean) < MIN_SCREEN_OBS or clean[feature].nunique() < 3:
        return pd.DataFrame()
    try:
        clean["bin"] = pd.qcut(clean[feature], q=min(q, clean[feature].nunique()), duplicates="drop")
    except ValueError:
        return pd.DataFrame()
    return clean.groupby("bin", observed=True).agg(
        n=(target_col, "size"),
        x_mean=(feature, "mean"),
        target_mean=(target_col, "mean"),
        target_abs_mean=(target_col, lambda s: s.abs().mean()),
    ).reset_index(drop=True)


def screen_feature(df, origin, target, feature):
    g = df.loc[df["origin"].eq(origin) & df["target_name"].eq(target)].copy()
    missing_rate = g[feature].isna().mean()
    clean = g[[feature, "target_residual"]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(clean) < MIN_SCREEN_OBS or clean[feature].nunique() < 3:
        return {"origin": origin, "target_name": target, "feature": feature, "n": len(clean), "missing_rate": missing_rate}
    slope, intercept, rvalue, pvalue, stderr = stats.linregress(clean[feature], clean["target_residual"])
    binned = decile_summary(clean, feature)
    return {
        "origin": origin,
        "target_name": target,
        "feature": feature,
        "n": len(clean),
        "corr_u_x": clean["target_residual"].corr(clean[feature]),
        "corr_abs_u_x": clean["target_residual"].abs().corr(clean[feature]),
        "slope": slope,
        "tstat": slope / stderr if pd.notna(stderr) and stderr > 0 else np.nan,
        "pvalue": pvalue,
        "r2": rvalue ** 2,
        "binned_spread": binned["target_mean"].max() - binned["target_mean"].min() if not binned.empty else np.nan,
        "missing_rate": missing_rate,
    }

screen_rows = []
for (origin, target), _ in modeling_panel.groupby(["origin", "target_name"]):
    for feature in SCREEN_FEATURES:
        screen_rows.append(screen_feature(modeling_panel, origin, target, feature))
faithful_feature_screening_summary = pd.DataFrame(screen_rows)
display(faithful_feature_screening_summary.sort_values(["origin", "target_name", "pvalue"], na_position="last"))

plot_specs = [
    ("after_friday", "OW", "WikiViewsSurprise", "u_OW vs WikiViewsSurprise"),
    ("after_saturday", "OW", "WikiViewsSurprise", "u_OW vs WikiViewsSurprise"),
    ("after_friday", "OW", "RelativeNewCompetition", "u_OW vs RelativeNewCompetition"),
    ("after_saturday", "OW", "RelativeNewCompetition", "u_OW vs RelativeNewCompetition"),
    ("after_saturday", "OW", "RelativeNewCompetition", "|u_OW| vs RelativeNewCompetition"),
    ("after_friday", "SunFri", "DirectNewCompetition", "daily-shape residual vs DirectNewCompetition"),
    ("after_saturday", "SunSat", "DirectNewCompetition", "daily-shape residual vs DirectNewCompetition"),
]

for origin, target, feature, title in plot_specs:
    subset = modeling_panel.loc[modeling_panel["origin"].eq(origin) & modeling_panel["target_name"].eq(target)].copy()
    if "|u" in title:
        subset["plot_target"] = subset["target_residual"].abs()
        target_col = "plot_target"
    else:
        target_col = "target_residual"
    binned = decile_summary(subset, feature, target_col=target_col)
    clean = subset[[feature, target_col]].dropna()
    if len(clean) < MIN_SCREEN_OBS or binned.empty:
        print(f"Skipping plot: {origin} {target} {feature}")
        continue
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    axes[0].scatter(clean[feature], clean[target_col], s=28, alpha=0.72)
    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set_title(f"{origin} {target}: {feature}")
    axes[0].set_xlabel(feature)
    axes[0].set_ylabel(target_col)
    axes[1].plot(binned["x_mean"], binned["target_mean"], marker="o")
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set_title(title + " deciles")
    axes[1].set_xlabel(feature)
    axes[1].set_ylabel("decile mean")
    plt.show()

## Rolling-Origin Residual Validation

Every candidate model is fit only on prior releases in the same origin and target. Feature standardization is also fit only on the training fold.

In [ ]:
def fit_linear_model(train, features):
    clean = train[["target_residual", *features]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(clean) < MIN_TRAIN_OBS:
        return None
    if not features:
        return {"features": [], "train_n": len(clean)}
    x = clean[features].to_numpy(dtype=float)
    y = clean["target_residual"].to_numpy(dtype=float)
    x_mean = x.mean(axis=0)
    x_sd = x.std(axis=0)
    x_sd = np.where((~np.isfinite(x_sd)) | (x_sd <= 1e-12), 1.0, x_sd)
    xz = (x - x_mean) / x_sd
    design = np.column_stack([np.ones(len(xz)), xz])
    penalty = RIDGE_LAMBDA * np.eye(design.shape[1])
    penalty[0, 0] = 0.0
    coef = np.linalg.solve(design.T @ design + penalty, design.T @ y)
    return {
        "features": list(features),
        "intercept": float(coef[0]),
        "coef": coef[1:],
        "x_mean": dict(zip(features, x_mean)),
        "x_sd": dict(zip(features, x_sd)),
        "train_n": len(clean),
    }


def predict_residual(row, fit):
    if fit is None:
        return np.nan
    if not fit.get("features"):
        return 0.0
    values = []
    for feature in fit["features"]:
        value = row.get(feature, np.nan)
        if pd.isna(value):
            return np.nan
        values.append((float(value) - fit["x_mean"][feature]) / fit["x_sd"][feature])
    return float(fit["intercept"] + np.dot(np.array(values), fit["coef"]))


def add_candidate_outcomes(row, u_hat):
    out = {}
    actual_ow = row["opening_weekend_gross_usd"]
    baseline_ow = row["baseline_pred_ow"]
    if row["target_name"] == "OW":
        candidate_ow = baseline_ow * np.exp(u_hat) if pd.notna(u_hat) and pd.notna(baseline_ow) else np.nan
        out["actual_value"] = actual_ow
        out["baseline_value"] = baseline_ow
        out["candidate_value"] = candidate_ow
    elif row["target_name"] == "SatFri":
        candidate_sat = row["friday_gross_usd"] * np.exp(row["baseline_y_sat_fri"] + u_hat) if pd.notna(u_hat) else np.nan
        candidate_ow = row["friday_gross_usd"] + candidate_sat + row["pred_sunday_gross_usd"]
        out["actual_value"] = row["saturday_gross_usd"]
        out["baseline_value"] = row["pred_saturday_gross_usd"]
        out["candidate_value"] = candidate_sat
    elif row["target_name"] == "SunFri":
        candidate_sun = row["friday_gross_usd"] * np.exp(row["baseline_y_sun_fri"] + u_hat) if pd.notna(u_hat) else np.nan
        candidate_ow = row["friday_gross_usd"] + row["pred_saturday_gross_usd"] + candidate_sun
        out["actual_value"] = row["sunday_gross_usd"]
        out["baseline_value"] = row["pred_sunday_gross_usd"]
        out["candidate_value"] = candidate_sun
    elif row["target_name"] == "SunSat":
        candidate_sun = row["saturday_gross_usd"] * np.exp(row["baseline_y_sun_sat"] + u_hat) if pd.notna(u_hat) else np.nan
        candidate_ow = row["friday_gross_usd"] + row["saturday_gross_usd"] + candidate_sun
        out["actual_value"] = row["sunday_gross_usd"]
        out["baseline_value"] = row["pred_sunday_gross_usd"]
        out["candidate_value"] = candidate_sun
    else:
        candidate_ow = np.nan
        out["actual_value"] = np.nan
        out["baseline_value"] = np.nan
        out["candidate_value"] = np.nan
    out["candidate_ow"] = candidate_ow
    out["baseline_ow"] = baseline_ow
    out["actual_ow"] = actual_ow
    return out


def rolling_predictions(df):
    rows = []
    for (origin, target), group in df.groupby(["origin", "target_name"]):
        group = group.sort_values(["opening_weekend_start", "release_run_id"]).copy()
        for model_name, features in MODEL_SPECS.items():
            for _, row in group.iterrows():
                if model_name == "M0_baseline":
                    u_hat = 0.0
                    train_n = len(group.loc[group["opening_weekend_start"].lt(row["opening_weekend_start"])])
                else:
                    train = group.loc[group["opening_weekend_start"].lt(row["opening_weekend_start"])]
                    fit = fit_linear_model(train, features)
                    u_hat = predict_residual(row, fit)
                    train_n = fit["train_n"] if fit is not None else len(train.dropna(subset=["target_residual", *features]))
                outcomes = add_candidate_outcomes(row, u_hat)
                candidate_error_log = row["target_residual"] - u_hat if pd.notna(row["target_residual"]) and pd.notna(u_hat) else np.nan
                baseline_error_log = row["target_residual"]
                ow_candidate_error_log = np.log(outcomes["actual_ow"] / outcomes["candidate_ow"]) if pd.notna(outcomes["actual_ow"]) and pd.notna(outcomes["candidate_ow"]) and outcomes["actual_ow"] > 0 and outcomes["candidate_ow"] > 0 else np.nan
                ow_baseline_error_log = np.log(outcomes["actual_ow"] / outcomes["baseline_ow"]) if pd.notna(outcomes["actual_ow"]) and pd.notna(outcomes["baseline_ow"]) and outcomes["actual_ow"] > 0 and outcomes["baseline_ow"] > 0 else np.nan
                rows.append({
                    "origin": origin,
                    "target_name": target,
                    "target_family": row["target_family"],
                    "candidate_model": model_name,
                    "candidate_features": ",".join(features),
                    "release_run_id": row["release_run_id"],
                    "title": row["title"],
                    "opening_weekend_start": row["opening_weekend_start"],
                    "release_year": row["release_year"],
                    "train_n": train_n,
                    "target_residual": row["target_residual"],
                    "predicted_residual": u_hat,
                    "baseline_error_log": baseline_error_log,
                    "candidate_error_log": candidate_error_log,
                    "actual_value": outcomes["actual_value"],
                    "baseline_value": outcomes["baseline_value"],
                    "candidate_value": outcomes["candidate_value"],
                    "actual_ow": outcomes["actual_ow"],
                    "baseline_ow": outcomes["baseline_ow"],
                    "candidate_ow": outcomes["candidate_ow"],
                    "ow_baseline_error_log": ow_baseline_error_log,
                    "ow_candidate_error_log": ow_candidate_error_log,
                })
    return pd.DataFrame(rows)

faithful_candidate_predictions = rolling_predictions(modeling_panel)
display(faithful_candidate_predictions.head())

In [ ]:
def rmse(series):
    clean = pd.to_numeric(pd.Series(series), errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    return float(np.sqrt(np.mean(np.square(clean)))) if len(clean) else np.nan


def summarize_predictions(predictions):
    rows = []
    for keys, g in predictions.groupby(["origin", "target_name", "target_family", "candidate_model"]):
        origin, target, family, model = keys
        scored = g.dropna(subset=["baseline_error_log", "candidate_error_log"]).copy()
        if scored.empty:
            rows.append({"origin": origin, "target_name": target, "target_family": family, "candidate_model": model, "n": 0})
            continue
        base_mae = scored["baseline_error_log"].abs().mean()
        cand_mae = scored["candidate_error_log"].abs().mean()
        row = {
            "origin": origin,
            "target_name": target,
            "target_family": family,
            "candidate_model": model,
            "candidate_features": scored["candidate_features"].iloc[0],
            "n": len(scored),
            "mean_train_n": scored["train_n"].mean(),
            "baseline_MAE_log_same_sample": base_mae,
            "candidate_MAE_log": cand_mae,
            "improvement_MAE_log_pct": (base_mae - cand_mae) / base_mae if base_mae > 0 else np.nan,
            "baseline_RMSE_log_same_sample": rmse(scored["baseline_error_log"]),
            "candidate_RMSE_log": rmse(scored["candidate_error_log"]),
            "baseline_ME_log_same_sample": scored["baseline_error_log"].mean(),
            "candidate_ME_log": scored["candidate_error_log"].mean(),
            "pct_movies_improved_abs_log_error": (scored["candidate_error_log"].abs() < scored["baseline_error_log"].abs()).mean(),
        }
        ow_scored = scored.dropna(subset=["ow_baseline_error_log", "ow_candidate_error_log"])
        if not ow_scored.empty:
            row["ow_baseline_MAE_log_same_sample"] = ow_scored["ow_baseline_error_log"].abs().mean()
            row["ow_candidate_MAE_log"] = ow_scored["ow_candidate_error_log"].abs().mean()
            row["ow_improvement_MAE_log_pct"] = (row["ow_baseline_MAE_log_same_sample"] - row["ow_candidate_MAE_log"]) / row["ow_baseline_MAE_log_same_sample"] if row["ow_baseline_MAE_log_same_sample"] > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["origin", "target_name", "candidate_MAE_log"], na_position="last")

faithful_rolling_model_comparison = summarize_predictions(faithful_candidate_predictions)
faithful_ow_model_comparison = faithful_rolling_model_comparison.loc[faithful_rolling_model_comparison["target_family"].eq("ow")].copy()
faithful_daily_shape_model_comparison = faithful_rolling_model_comparison.loc[faithful_rolling_model_comparison["target_family"].eq("daily_shape")].copy()

display(faithful_ow_model_comparison)
display(faithful_daily_shape_model_comparison.head(40))

## Focused W3 After-Saturday Validation

This section validates whether `W3 = WikiViewsSurprise + WikiGrowth` is genuinely useful for the after-Saturday `Sun/Sat` residual and, secondarily, after-Saturday OW. It uses only the exact shared sample where `M0`, `W1`, and `W3` all have valid rolling-origin predictions.

Promotion rule:

- `SunSat` MAE log improves by at least 3%;
- OW MAE log improves by at least 2%;
- percent improved is at least 50%;
- no obvious slice blow-up across year, corridor, consensus-size decile, or WikiViewsSurprise decile.


In [ ]:
W3_VALIDATION_MODELS = [
    "M0_baseline",
    "W1_wiki_views_surprise",
    "W3_wiki_surprise_plus_growth",
]
W3_VALIDATION_ORIGIN = "after_saturday"
W3_VALIDATION_TARGET = "SunSat"

w3_candidate = faithful_candidate_predictions.loc[
    faithful_candidate_predictions["origin"].eq(W3_VALIDATION_ORIGIN)
    & faithful_candidate_predictions["target_name"].eq(W3_VALIDATION_TARGET)
    & faithful_candidate_predictions["candidate_model"].eq("W3_wiki_surprise_plus_growth")
    & faithful_candidate_predictions["candidate_error_log"].notna()
    & faithful_candidate_predictions["ow_candidate_error_log"].notna()
].copy()

valid_sets = []
for model_name in W3_VALIDATION_MODELS:
    model_rows = faithful_candidate_predictions.loc[
        faithful_candidate_predictions["origin"].eq(W3_VALIDATION_ORIGIN)
        & faithful_candidate_predictions["target_name"].eq(W3_VALIDATION_TARGET)
        & faithful_candidate_predictions["candidate_model"].eq(model_name)
        & faithful_candidate_predictions["candidate_error_log"].notna()
        & faithful_candidate_predictions["ow_candidate_error_log"].notna()
    ]
    valid_sets.append(set(model_rows["release_run_id"]))

w3_exact_ids = sorted(set.intersection(*valid_sets))
w3_validation_predictions = faithful_candidate_predictions.loc[
    faithful_candidate_predictions["origin"].eq(W3_VALIDATION_ORIGIN)
    & faithful_candidate_predictions["target_name"].eq(W3_VALIDATION_TARGET)
    & faithful_candidate_predictions["candidate_model"].isin(W3_VALIDATION_MODELS)
    & faithful_candidate_predictions["release_run_id"].isin(w3_exact_ids)
].copy()

metadata_cols = [
    "origin", "target_name", "release_run_id", "release_year", "release_corridor", "scale_prior_usd",
    "WikiViewsSurprise", "WikiGrowth", "WikiViews", "title", "opening_weekend_start",
]
w3_validation_metadata = modeling_panel.loc[
    modeling_panel["origin"].eq(W3_VALIDATION_ORIGIN)
    & modeling_panel["target_name"].eq(W3_VALIDATION_TARGET)
    & modeling_panel["release_run_id"].isin(w3_exact_ids),
    [c for c in metadata_cols if c in modeling_panel.columns],
].drop_duplicates("release_run_id").copy()

w3_validation_predictions = w3_validation_predictions.merge(
    w3_validation_metadata.drop(columns=["origin", "target_name", "title", "opening_weekend_start"], errors="ignore"),
    on="release_run_id",
    how="left",
    suffixes=("", "_meta"),
)
w3_validation_predictions["release_year"] = w3_validation_predictions["release_year_meta"].where(
    w3_validation_predictions.get("release_year_meta").notna(),
    w3_validation_predictions["release_year"],
) if "release_year_meta" in w3_validation_predictions.columns else w3_validation_predictions["release_year"]
w3_validation_predictions = w3_validation_predictions.drop(columns=["release_year_meta"], errors="ignore")

baseline_lookup = w3_validation_predictions.loc[
    w3_validation_predictions["candidate_model"].eq("M0_baseline"),
    ["release_run_id", "candidate_error_log", "ow_candidate_error_log"],
].rename(columns={
    "candidate_error_log": "m0_sunsat_error_log",
    "ow_candidate_error_log": "m0_ow_error_log",
})
w3_validation_predictions = w3_validation_predictions.merge(baseline_lookup, on="release_run_id", how="left")


def validation_metric_row(df, model_name):
    g = df.loc[df["candidate_model"].eq(model_name)].copy()
    return {
        "origin": W3_VALIDATION_ORIGIN,
        "target_name": W3_VALIDATION_TARGET,
        "candidate_model": model_name,
        "candidate_features": g["candidate_features"].iloc[0] if len(g) else "",
        "n": len(g),
        "MAE_log_SunSat": g["candidate_error_log"].abs().mean(),
        "RMSE_log_SunSat": rmse(g["candidate_error_log"]),
        "ME_log_SunSat": g["candidate_error_log"].mean(),
        "MAE_log_OW": g["ow_candidate_error_log"].abs().mean(),
        "RMSE_log_OW": rmse(g["ow_candidate_error_log"]),
        "ME_log_OW": g["ow_candidate_error_log"].mean(),
        "SunSat_MAE_improvement_vs_M0_pct": (
            g["m0_sunsat_error_log"].abs().mean() - g["candidate_error_log"].abs().mean()
        ) / g["m0_sunsat_error_log"].abs().mean() if len(g) and g["m0_sunsat_error_log"].abs().mean() > 0 else np.nan,
        "OW_MAE_improvement_vs_M0_pct": (
            g["m0_ow_error_log"].abs().mean() - g["ow_candidate_error_log"].abs().mean()
        ) / g["m0_ow_error_log"].abs().mean() if len(g) and g["m0_ow_error_log"].abs().mean() > 0 else np.nan,
        "pct_improved_SunSat": (g["candidate_error_log"].abs() < g["m0_sunsat_error_log"].abs()).mean() if len(g) else np.nan,
        "pct_improved_OW": (g["ow_candidate_error_log"].abs() < g["m0_ow_error_log"].abs()).mean() if len(g) else np.nan,
    }

faithful_w3_validation_comparison = pd.DataFrame([
    validation_metric_row(w3_validation_predictions, model_name) for model_name in W3_VALIDATION_MODELS
])
display(faithful_w3_validation_comparison)


def add_decile(series, label):
    clean = pd.to_numeric(series, errors="coerce")
    try:
        return pd.qcut(clean, q=min(10, clean.nunique(dropna=True)), duplicates="drop")
    except ValueError:
        return pd.Series(pd.NA, index=series.index, dtype="object")

slice_meta = w3_validation_metadata.copy()
consensus_size_log = np.log1p(pd.Series(pd.to_numeric(slice_meta.get("scale_prior_usd"), errors="coerce"), index=slice_meta.index).clip(lower=0))
slice_meta["consensus_size_decile"] = add_decile(consensus_size_log, "consensus")
slice_meta["wiki_surprise_decile"] = add_decile(slice_meta.get("WikiViewsSurprise"), "wiki_surprise")

slice_predictions = w3_validation_predictions.merge(
    slice_meta[["release_run_id", "consensus_size_decile", "wiki_surprise_decile"]],
    on="release_run_id",
    how="left",
)

slice_specs = [
    ("year", "release_year"),
    ("corridor", "release_corridor"),
    ("consensus_size_decile", "consensus_size_decile"),
    ("wiki_surprise_decile", "wiki_surprise_decile"),
]

slice_rows = []
for slice_family, col in slice_specs:
    if col not in slice_predictions.columns:
        continue
    for slice_value, g_all in slice_predictions.groupby(col, dropna=False, observed=True):
        baseline = g_all.loc[g_all["candidate_model"].eq("M0_baseline")]
        w1 = g_all.loc[g_all["candidate_model"].eq("W1_wiki_views_surprise")]
        w3 = g_all.loc[g_all["candidate_model"].eq("W3_wiki_surprise_plus_growth")]
        if baseline.empty or w3.empty:
            continue
        row = {
            "slice_family": slice_family,
            "slice_value": str(slice_value),
            "n": len(w3),
            "M0_MAE_log_SunSat": baseline["candidate_error_log"].abs().mean(),
            "W1_MAE_log_SunSat": w1["candidate_error_log"].abs().mean() if not w1.empty else np.nan,
            "W3_MAE_log_SunSat": w3["candidate_error_log"].abs().mean(),
            "W3_SunSat_improvement_pct": (baseline["candidate_error_log"].abs().mean() - w3["candidate_error_log"].abs().mean()) / baseline["candidate_error_log"].abs().mean() if baseline["candidate_error_log"].abs().mean() > 0 else np.nan,
            "M0_MAE_log_OW": baseline["ow_candidate_error_log"].abs().mean(),
            "W1_MAE_log_OW": w1["ow_candidate_error_log"].abs().mean() if not w1.empty else np.nan,
            "W3_MAE_log_OW": w3["ow_candidate_error_log"].abs().mean(),
            "W3_OW_improvement_pct": (baseline["ow_candidate_error_log"].abs().mean() - w3["ow_candidate_error_log"].abs().mean()) / baseline["ow_candidate_error_log"].abs().mean() if baseline["ow_candidate_error_log"].abs().mean() > 0 else np.nan,
            "W3_pct_improved_SunSat": (w3["candidate_error_log"].abs().to_numpy() < baseline["candidate_error_log"].abs().to_numpy()).mean() if len(w3) == len(baseline) else np.nan,
            "W3_pct_improved_OW": (w3["ow_candidate_error_log"].abs().to_numpy() < baseline["ow_candidate_error_log"].abs().to_numpy()).mean() if len(w3) == len(baseline) else np.nan,
        }
        slice_rows.append(row)

faithful_w3_validation_slices = pd.DataFrame(slice_rows)
display(faithful_w3_validation_slices.sort_values(["slice_family", "slice_value"]))

w3_overall = faithful_w3_validation_comparison.loc[
    faithful_w3_validation_comparison["candidate_model"].eq("W3_wiki_surprise_plus_growth")
].iloc[0]
large_slices = faithful_w3_validation_slices.loc[faithful_w3_validation_slices["n"].ge(5)].copy()
large_slices["sun_blowup"] = large_slices["W3_SunSat_improvement_pct"].lt(-0.05)
large_slices["ow_blowup"] = large_slices["W3_OW_improvement_pct"].lt(-0.05)
faithful_w3_promotion_check = pd.DataFrame([{
    "origin": W3_VALIDATION_ORIGIN,
    "target_name": W3_VALIDATION_TARGET,
    "candidate_model": "W3_wiki_surprise_plus_growth",
    "n_exact_sample": int(w3_overall["n"]),
    "passes_SunSat_MAE_3pct": bool(w3_overall["SunSat_MAE_improvement_vs_M0_pct"] >= 0.03),
    "passes_OW_MAE_2pct": bool(w3_overall["OW_MAE_improvement_vs_M0_pct"] >= 0.02),
    "passes_pct_improved_50pct_SunSat": bool(w3_overall["pct_improved_SunSat"] >= 0.50),
    "passes_pct_improved_50pct_OW": bool(w3_overall["pct_improved_OW"] >= 0.50),
    "large_slice_count": int(len(large_slices)),
    "large_slice_sunsat_blowup_count": int(large_slices["sun_blowup"].sum()),
    "large_slice_ow_blowup_count": int(large_slices["ow_blowup"].sum()),
    "promotion_label_if_passed": "AS0+W3 Sunday shape adjustment",
    "promote": bool(
        (w3_overall["SunSat_MAE_improvement_vs_M0_pct"] >= 0.03)
        and (w3_overall["OW_MAE_improvement_vs_M0_pct"] >= 0.02)
        and (w3_overall["pct_improved_SunSat"] >= 0.50)
        and (w3_overall["pct_improved_OW"] >= 0.50)
        and (large_slices["sun_blowup"].sum() == 0)
        and (large_slices["ow_blowup"].sum() == 0)
    ),
    "competition_next_step": "Holdover attraction was rebuilt in this notebook using opening-weekend gross decay proxy; replace with true prior-weekend gross/current theaters when that feed is available.",
}])
display(faithful_w3_promotion_check)


## Path B: Pre-Weekend Wiki Pseudo-Prior

This section uses the paper-style pre-weekend logic where it fits `Wiki + theaters + release corridor -> log(OW)` in rolling chronological validation.

It compares four prior policies:

- `Consensus`: available consensus estimate only;
- `PseudoPrior`: rolling Wiki/theaters/corridor forecast;
- `Consensus+PseudoPrior`: rolling blend of consensus and pseudo-prior on rows where both exist;
- `NoConsensusFallback`: consensus where available, otherwise pseudo-prior.


In [ ]:
def assign_release_corridor(df):
    out = df.copy()
    start = pd.to_datetime(out["opening_weekend_start"], errors="coerce")
    month = pd.to_numeric(out.get("release_month", start.dt.month), errors="coerce")
    iso_week = pd.to_numeric(out.get("release_week_of_year", start.dt.isocalendar().week), errors="coerce")
    day = start.dt.day

    christmas = (month.eq(12) & iso_week.ge(50)) | (month.eq(1) & iso_week.le(1))
    thanksgiving = month.eq(11) & iso_week.between(46, 48)
    memorial_july4 = iso_week.between(21, 27) & month.isin([5, 6, 7])
    halloween = month.eq(10) & day.between(14, 31)
    summer = month.isin([6, 7, 8])
    fall = month.isin([9, 10, 11])

    out["release_corridor"] = np.select(
        [christmas, thanksgiving, memorial_july4, halloween, summer, fall],
        ["Christmas/NewYear", "Thanksgiving", "MemorialDay/July4", "Halloween", "summer", "fall"],
        default="ordinary",
    )
    return out


def build_pseudo_prior_panel(shape_path, consensus_path, wiki_features):
    shape_cols = [
        "release_run_id", "movie_id", "title", "opening_weekend_start", "release_year", "release_month",
        "release_week_of_year", "season_bucket", "opening_weekend_theaters", "opening_weekend_gross_usd",
        "latest_estimate_mid_usd", "wiki_views_cume_to_m1", "genre", "broad_segment", "segment_current_broad",
    ]
    panel = pd.read_csv(shape_path, usecols=lambda c: c in shape_cols)
    panel["release_run_id"] = pd.to_numeric(panel["release_run_id"], errors="coerce")
    panel["opening_weekend_start"] = pd.to_datetime(panel["opening_weekend_start"], errors="coerce")
    for col in ["release_year", "release_month", "release_week_of_year", "opening_weekend_theaters", "opening_weekend_gross_usd", "latest_estimate_mid_usd", "wiki_views_cume_to_m1"]:
        if col in panel.columns:
            panel[col] = pd.to_numeric(panel[col], errors="coerce")
    panel = assign_release_corridor(panel)

    if consensus_path.exists():
        consensus = pd.read_csv(consensus_path, usecols=lambda c: c in ["release_run_id", "consensus_estimate_usd"])
        consensus["release_run_id"] = pd.to_numeric(consensus["release_run_id"], errors="coerce")
        consensus["consensus_estimate_usd"] = pd.to_numeric(consensus["consensus_estimate_usd"], errors="coerce")
        consensus = consensus.drop_duplicates("release_run_id")
        panel = panel.merge(consensus, on="release_run_id", how="left")
    else:
        panel["consensus_estimate_usd"] = np.nan

    panel["consensus_estimate_usd"] = panel["consensus_estimate_usd"].where(
        panel["consensus_estimate_usd"].notna(),
        pd.to_numeric(panel.get("latest_estimate_mid_usd"), errors="coerce"),
    )

    wiki_keep = ["release_run_id", "WikiViews", "WikiUsers", "WikiEdits", "WikiRigor", "WikiGrowth"]
    panel = panel.merge(wiki_features[[c for c in wiki_keep if c in wiki_features.columns]], on="release_run_id", how="left")
    if "WikiViews" not in panel.columns:
        panel["WikiViews"] = np.nan
    panel["WikiViews"] = panel["WikiViews"].where(
        panel["WikiViews"].notna(),
        np.log1p(pd.to_numeric(panel.get("wiki_views_cume_to_m1"), errors="coerce").clip(lower=0)),
    )
    panel["log_theaters"] = np.log1p(pd.to_numeric(panel.get("opening_weekend_theaters"), errors="coerce").clip(lower=0))
    panel["log_actual_ow"] = np.log(pd.to_numeric(panel["opening_weekend_gross_usd"], errors="coerce"))
    panel["log_consensus"] = np.log(pd.to_numeric(panel["consensus_estimate_usd"], errors="coerce"))
    panel = panel.replace([np.inf, -np.inf], np.nan)
    return panel.dropna(subset=["release_run_id", "opening_weekend_start", "log_actual_ow"]).sort_values(["opening_weekend_start", "release_run_id"]).reset_index(drop=True)


def fit_design_model(train, numeric_cols, categorical_cols, target_col="log_actual_ow"):
    cols = [target_col, *numeric_cols, *categorical_cols]
    clean = train[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(clean) < MIN_TRAIN_OBS:
        return None
    categories = {col: sorted(clean[col].astype(str).unique()) for col in categorical_cols}
    x_parts = []
    feature_names = []
    for col in numeric_cols:
        x_parts.append(clean[[col]].to_numpy(dtype=float))
        feature_names.append(col)
    for col in categorical_cols:
        values = clean[col].astype(str)
        for cat in categories[col][1:]:
            x_parts.append(values.eq(cat).astype(float).to_numpy()[:, None])
            feature_names.append(f"{col}={cat}")
    x = np.hstack(x_parts) if x_parts else np.empty((len(clean), 0))
    x_mean = x.mean(axis=0) if x.shape[1] else np.array([])
    x_sd = x.std(axis=0) if x.shape[1] else np.array([])
    x_sd = np.where((~np.isfinite(x_sd)) | (x_sd <= 1e-12), 1.0, x_sd)
    xz = (x - x_mean) / x_sd if x.shape[1] else x
    design = np.column_stack([np.ones(len(clean)), xz])
    y = clean[target_col].to_numpy(dtype=float)
    penalty = RIDGE_LAMBDA * np.eye(design.shape[1])
    penalty[0, 0] = 0.0
    coef = np.linalg.solve(design.T @ design + penalty, design.T @ y)
    return {
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "categories": categories,
        "feature_names": feature_names,
        "x_mean": x_mean,
        "x_sd": x_sd,
        "coef": coef,
        "train_n": len(clean),
    }


def predict_design_model(row, fit):
    if fit is None:
        return np.nan
    values = []
    for col in fit["numeric_cols"]:
        value = row.get(col, np.nan)
        if pd.isna(value):
            return np.nan
        values.append(float(value))
    for col in fit["categorical_cols"]:
        value = str(row.get(col, ""))
        for cat in fit["categories"][col][1:]:
            values.append(1.0 if value == cat else 0.0)
    x = np.array(values, dtype=float)
    if len(x):
        x = (x - fit["x_mean"]) / fit["x_sd"]
        return float(fit["coef"][0] + np.dot(x, fit["coef"][1:]))
    return float(fit["coef"][0])


def add_rolling_pseudo_prior_predictions(panel):
    out = panel.copy()
    out["pseudo_prior_log_ow"] = np.nan
    out["pseudo_prior_train_n"] = 0
    out["consensus_plus_pseudo_log_ow"] = np.nan
    out["consensus_plus_pseudo_train_n"] = 0
    pseudo_numeric = ["WikiViews", "log_theaters"]
    pseudo_categorical = ["release_corridor"]
    blend_numeric = ["log_consensus", "pseudo_prior_log_ow"]

    for row_idx, row in out.sort_values(["opening_weekend_start", "release_run_id"]).iterrows():
        train = out.loc[out["opening_weekend_start"].lt(row["opening_weekend_start"])].copy()
        pseudo_fit = fit_design_model(train, pseudo_numeric, pseudo_categorical)
        out.loc[row_idx, "pseudo_prior_log_ow"] = predict_design_model(row, pseudo_fit)
        out.loc[row_idx, "pseudo_prior_train_n"] = pseudo_fit["train_n"] if pseudo_fit is not None else len(train.dropna(subset=["log_actual_ow", *pseudo_numeric, *pseudo_categorical]))

        train_with_pseudo = out.loc[out["opening_weekend_start"].lt(row["opening_weekend_start"])].copy()
        blend_fit = fit_design_model(train_with_pseudo, blend_numeric, [], target_col="log_actual_ow")
        out.loc[row_idx, "consensus_plus_pseudo_log_ow"] = predict_design_model(out.loc[row_idx], blend_fit)
        out.loc[row_idx, "consensus_plus_pseudo_train_n"] = blend_fit["train_n"] if blend_fit is not None else len(train_with_pseudo.dropna(subset=["log_actual_ow", *blend_numeric]))
    return out


def build_prior_policy_predictions(panel):
    rows = []
    for _, row in panel.iterrows():
        policy_logs = {
            "Consensus": row.get("log_consensus", np.nan),
            "PseudoPrior": row.get("pseudo_prior_log_ow", np.nan),
            "Consensus+PseudoPrior": row.get("consensus_plus_pseudo_log_ow", np.nan),
            "NoConsensusFallback": row.get("log_consensus", np.nan) if pd.notna(row.get("log_consensus", np.nan)) else row.get("pseudo_prior_log_ow", np.nan),
        }
        for model, pred_log in policy_logs.items():
            pred_usd = float(np.exp(pred_log)) if pd.notna(pred_log) else np.nan
            actual_usd = row.get("opening_weekend_gross_usd", np.nan)
            rows.append({
                "prior_model": model,
                "release_run_id": row["release_run_id"],
                "title": row["title"],
                "opening_weekend_start": row["opening_weekend_start"],
                "release_year": row["release_year"],
                "release_corridor": row["release_corridor"],
                "has_consensus": pd.notna(row.get("log_consensus", np.nan)),
                "has_pseudo_prior": pd.notna(row.get("pseudo_prior_log_ow", np.nan)),
                "actual_opening_weekend_gross_usd": actual_usd,
                "pred_opening_weekend_gross_usd": pred_usd,
                "error_log": row["log_actual_ow"] - pred_log if pd.notna(pred_log) else np.nan,
                "abs_pct_error": abs(actual_usd - pred_usd) / actual_usd if pd.notna(actual_usd) and pd.notna(pred_usd) and actual_usd > 0 else np.nan,
                "pseudo_prior_train_n": row.get("pseudo_prior_train_n", np.nan),
                "consensus_plus_pseudo_train_n": row.get("consensus_plus_pseudo_train_n", np.nan),
            })
    return pd.DataFrame(rows)


def summarize_prior_policy_predictions(predictions):
    rows = []
    consensus_error_by_release = predictions.loc[
        predictions["prior_model"].eq("Consensus"), ["release_run_id", "error_log"]
    ].dropna().drop_duplicates("release_run_id").set_index("release_run_id")["error_log"]
    sample_masks = {
        "available_policy_rows": predictions["error_log"].notna(),
        "common_consensus_pseudo": predictions["has_consensus"].fillna(False) & predictions["has_pseudo_prior"].fillna(False) & predictions["error_log"].notna(),
        "no_consensus_only": (~predictions["has_consensus"].fillna(False)) & predictions["error_log"].notna(),
    }
    for sample, mask in sample_masks.items():
        for model, g in predictions.loc[mask].groupby("prior_model"):
            scored = g.dropna(subset=["error_log"]).copy()
            if scored.empty:
                continue
            scored["consensus_error_same_release"] = scored["release_run_id"].map(consensus_error_by_release)
            same_release = scored.dropna(subset=["consensus_error_same_release"])
            consensus_mae_same = same_release["consensus_error_same_release"].abs().mean() if not same_release.empty else np.nan
            candidate_mae_same = same_release["error_log"].abs().mean() if not same_release.empty else np.nan
            rows.append({
                "sample": sample,
                "prior_model": model,
                "n": len(scored),
                "n_consensus_same_release": len(same_release),
                "first_opening": scored["opening_weekend_start"].min(),
                "last_opening": scored["opening_weekend_start"].max(),
                "ME_log": scored["error_log"].mean(),
                "MAE_log": scored["error_log"].abs().mean(),
                "RMSE_log": rmse(scored["error_log"]),
                "MdAPE": scored["abs_pct_error"].median(),
                "mean_APE": scored["abs_pct_error"].mean(),
                "consensus_MAE_log_same_release": consensus_mae_same,
                "candidate_MAE_log_same_release": candidate_mae_same,
                "MAE_log_improvement_vs_consensus_pct": (consensus_mae_same - candidate_mae_same) / consensus_mae_same if pd.notna(consensus_mae_same) and consensus_mae_same > 0 else np.nan,
            })
    summary = pd.DataFrame(rows)
    if summary.empty:
        return summary
    return summary.sort_values(["sample", "MAE_log"])

pseudo_prior_panel = add_rolling_pseudo_prior_predictions(build_pseudo_prior_panel(SHAPE_BASE_PATH, CONSENSUS_BENCHMARK_PATH, wiki_raw))
faithful_pseudo_prior_predictions = build_prior_policy_predictions(pseudo_prior_panel)
faithful_pseudo_prior_model_comparison = summarize_prior_policy_predictions(faithful_pseudo_prior_predictions)

pseudo_prior_coverage = pd.DataFrame([{
    "rows": len(pseudo_prior_panel),
    "has_wiki_views": int(pseudo_prior_panel["WikiViews"].notna().sum()),
    "has_theaters": int(pseudo_prior_panel["log_theaters"].notna().sum()),
    "has_consensus": int(pseudo_prior_panel["log_consensus"].notna().sum()),
    "has_pseudo_prior_prediction": int(pseudo_prior_panel["pseudo_prior_log_ow"].notna().sum()),
    "has_consensus_plus_pseudo_prediction": int(pseudo_prior_panel["consensus_plus_pseudo_log_ow"].notna().sum()),
}])
display(pseudo_prior_coverage)
display(faithful_pseudo_prior_model_comparison)


## Promotion Review

In [ ]:
def build_recommendations(summary):
    rows = []
    for (origin, target, family), g in summary.groupby(["origin", "target_name", "target_family"]):
        candidates = g.loc[g["candidate_model"].ne("M0_baseline")].copy()
        candidates = candidates.loc[candidates["n"].fillna(0).ge(MIN_SCREEN_OBS)]
        if candidates.empty:
            rows.append({"origin": origin, "target_name": target, "target_family": family, "recommendation": "hold", "reason": "No candidate has enough scored rows."})
            continue
        candidates["abs_me_worsening"] = candidates["candidate_ME_log"].abs() - candidates["baseline_ME_log_same_sample"].abs()
        if family == "ow":
            eligible = candidates.loc[
                candidates["improvement_MAE_log_pct"].ge(0.02)
                & candidates["abs_me_worsening"].le(0.01)
                & candidates["pct_movies_improved_abs_log_error"].ge(0.50)
            ].sort_values(["improvement_MAE_log_pct", "candidate_MAE_log"], ascending=[False, True])
            best = candidates.sort_values(["improvement_MAE_log_pct", "candidate_MAE_log"], ascending=[False, True]).iloc[0]
            if eligible.empty:
                rows.append({
                    "origin": origin,
                    "target_name": target,
                    "target_family": family,
                    "recommendation": "reject_for_now",
                    "selected_model": best["candidate_model"],
                    "improvement_MAE_log_pct": best["improvement_MAE_log_pct"],
                    "pct_movies_improved_abs_log_error": best["pct_movies_improved_abs_log_error"],
                    "reason": "No OW candidate clears the 2-3% MAE_log, signed-error, and >=50% movie-improvement rules.",
                })
            else:
                selected = eligible.iloc[0]
                rows.append({
                    "origin": origin,
                    "target_name": target,
                    "target_family": family,
                    "recommendation": "hold_for_review",
                    "selected_model": selected["candidate_model"],
                    "selected_features": selected["candidate_features"],
                    "improvement_MAE_log_pct": selected["improvement_MAE_log_pct"],
                    "pct_movies_improved_abs_log_error": selected["pct_movies_improved_abs_log_error"],
                    "reason": "Numeric OW screen clears; inspect stability and feature interpretation before promotion.",
                })
        else:
            candidates["ow_worsening"] = candidates["ow_candidate_MAE_log"] - candidates["ow_baseline_MAE_log_same_sample"]
            eligible = candidates.loc[
                candidates["improvement_MAE_log_pct"].ge(0.02)
                & candidates["ow_worsening"].le(0.002)
            ].sort_values(["improvement_MAE_log_pct", "candidate_MAE_log"], ascending=[False, True])
            best = candidates.sort_values(["improvement_MAE_log_pct", "candidate_MAE_log"], ascending=[False, True]).iloc[0]
            rows.append({
                "origin": origin,
                "target_name": target,
                "target_family": family,
                "recommendation": "diagnostic_only" if not eligible.empty else "reject_for_now",
                "selected_model": (eligible.iloc[0] if not eligible.empty else best)["candidate_model"],
                "improvement_MAE_log_pct": (eligible.iloc[0] if not eligible.empty else best)["improvement_MAE_log_pct"],
                "ow_improvement_MAE_log_pct": (eligible.iloc[0] if not eligible.empty else best).get("ow_improvement_MAE_log_pct", np.nan),
                "reason": "Daily-shape model is diagnostic; promote only if daily target improves without OW degradation.",
            })
    return pd.DataFrame(rows)

faithful_model_recommendation = build_recommendations(faithful_rolling_model_comparison)
display(faithful_model_recommendation)

## Export Outputs

In [ ]:
DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

exports = {
    DIAGNOSTICS_DIR / "faithful_external_feature_coverage.csv": faithful_feature_coverage,
    DIAGNOSTICS_DIR / "faithful_external_feature_screening_summary.csv": faithful_feature_screening_summary,
    DIAGNOSTICS_DIR / "faithful_external_rolling_model_comparison.csv": faithful_rolling_model_comparison,
    DIAGNOSTICS_DIR / "faithful_external_ow_model_comparison.csv": faithful_ow_model_comparison,
    DIAGNOSTICS_DIR / "faithful_external_daily_shape_model_comparison.csv": faithful_daily_shape_model_comparison,
    DIAGNOSTICS_DIR / "faithful_external_model_recommendation.csv": faithful_model_recommendation,
    DIAGNOSTICS_DIR / "faithful_w3_after_saturday_exact_sample_comparison.csv": faithful_w3_validation_comparison,
    DIAGNOSTICS_DIR / "faithful_w3_after_saturday_stability_slices.csv": faithful_w3_validation_slices,
    DIAGNOSTICS_DIR / "faithful_w3_after_saturday_promotion_check.csv": faithful_w3_promotion_check,
    DIAGNOSTICS_DIR / "faithful_pseudo_prior_model_comparison.csv": faithful_pseudo_prior_model_comparison,
    PREDICTIONS_DIR / "faithful_external_candidate_predictions.csv": faithful_candidate_predictions,
    PREDICTIONS_DIR / "faithful_pseudo_prior_predictions.csv": faithful_pseudo_prior_predictions,
}
for path, table in exports.items():
    table.to_csv(path, index=False)

pd.DataFrame({"artifact": [path.name for path in exports], "path": [str(path) for path in exports]})
